# Taller · Una recta no se dobla

El viernes Daniela prometió 14.420 unidades al comité de Bebidas Cumbre usando el plano de TV y radio. Hoy revisas esa cifra, le enseñas al modelo a doblarse y aprendes a frenarlo cuando se dobla de más. Este cuaderno sigue las seis etapas de la página de D2L. No necesitas escribir Python: pídele cada celda a Claude, Gemini o Antigravity. Lo que se califica es que sepas qué te dio y lo hayas verificado.

> **Hoy haces** · Encuentras dónde falla el plano, agregas curvas e interacciones, corriges la promesa, contestas la oferta de la agencia y pruebas Ridge y Lasso.
>
> **Entrega** · Enlace público de este cuaderno con todo ejecutado, memo de una página y `prompts.md`.
>
> **Antes de empezar** · En Colab, arrastra `mercados-bebidas-cumbre.csv` y `consumo-combustible-co2.csv` al panel de archivos de la izquierda.

In [ ]:
# --- Setup ---
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error as mae, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

df = pd.read_csv("mercados-bebidas-cumbre.csv")
MARGEN = 12  # USD de margen por unidad. Ventas en miles de unidades, inversión en miles de USD.

# Train y test: siempre la misma división, la del caso anterior
tr, te = train_test_split(df, test_size=0.25, random_state=42)
MEDIOS = ["tv", "radio", "prensa"]
print(df.shape, tr.shape, te.shape)

## Etapa 1 · Dónde falla el plano

Ajusta el plano `ventas ~ tv + radio` con los 200 mercados. Calcula el error de cada mercado (real menos pronóstico), dibújalo en un gráfico de TV contra radio y promedia los mercados con `tv <= 90` y `radio >= 35`.

In [ ]:
# Pega aquí el código que te entregue Claude, Gemini o Antigravity.
# Pista de prompt: ajusta ventas contra tv y radio con los 200 mercados, calcula el error de cada uno
# y dibuja tv contra radio con el color según ese error. Luego promedia los mercados con tv <= 90 y radio >= 35.

**Decisión para el memo.** ¿El error del plano es ruido o patrón? Si es patrón, ¿en qué tipo de mercado promete de más?

## Etapa 2 · Doblar el plano

Agrega `tv × radio` y después `tv²`. Mide cada modelo con los 50 mercados de prueba y compáralo con la línea base.

In [ ]:
def medir(modelo, cols, nombre):
    """Entrena con tr, mide error medio absoluto en te."""
    modelo.fit(tr[cols], tr.ventas)
    print(f"{nombre:28s} error prueba {mae(te.ventas, modelo.predict(te[cols])):.3f}")

for parte in (tr, te):
    parte.loc[:, "tv_radio"] = parte.tv * parte.radio

# Pega aquí el código que te entregue tu asistente.
# Pista de prompt: usa las variables tr (train) y te (test). Crea tv al cuadrado y mide con la función medir
# cuatro modelos: el promedio, tv y radio, con tv por radio, y con tv por radio y tv al cuadrado.

**Decisión para el memo.** ¿Cuál de las dos columnas aportó más? ¿Por qué tiene sentido de negocio?

## Etapa 3 · Corregir la promesa

Ajusta el modelo de cuatro columnas con los 200 mercados. Pronostica los dos planes y calcula la utilidad: unidades por margen menos inversión. Luego calcula cuánto rinde la radio cuando la TV vale 50, 150 y 250.

In [ ]:
# Pega aquí el código que te entregue tu asistente.
# Pista de prompt: ajusta el modelo de cuatro columnas con los 200 mercados y pronostica TV 150 con radio 25,
# y TV 50 con radio 49. Todo está en miles y cada unidad deja 12 USD. Dame unidades, margen, inversión y utilidad.
# Luego dime cuántas unidades trae cada mil USD en radio con TV de 50, 150 y 250.

**Decisión para el memo.** ¿Cambia la decisión de lanzar con el plan de Daniela? ¿Qué cifra le dices al comité y cómo explicas que bajó?

## Etapa 4 · La oferta de la agencia

Ajusta polinomios de grado 1 a 7 con los tres medios y mira el R² de entrenamiento junto al error de prueba.

In [ ]:
# Pega aquí el código que te entregue tu asistente.
# Pista de prompt: con los tres medios, ajusta polinomios de grado 1 a 7 usando solo train. Dame una tabla con
# columnas, R² de train, error de train y error de test. Luego el error del grado 6 mercado por mercado.

**Decisión para el memo.** ¿Compran el modelo de la agencia? Respondan con el número que un comité entendería.

## Etapa 5 · El freno

Grado 5 con los tres medios. Elige α con validación cruzada dentro de los 150 de entrenamiento. Recién después abre los 50 de prueba.

In [ ]:
kf = KFold(5, shuffle=True, random_state=42)
ALPHAS_LASSO = [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
ALPHAS_RIDGE = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]

def freno(modelo):
    return make_pipeline(PolynomialFeatures(5, include_bias=False), StandardScaler(), modelo)

# Pega aquí el código que te entregue tu asistente.
# Pista de prompt: para cada alfa de las listas, calcula el error absoluto medio con validación cruzada de cinco
# partes, solo dentro de train. Elige el mejor alfa de Lasso y de Ridge. Al final mide en test y dime cuántas
# columnas quedan vivas. Revisa: pregúntale si usó test para elegir alfa.

**Decisión para el memo.** ¿Con cuál se queda Daniela: la curva de cuatro columnas o el Lasso? Argumenten con error y con lo que se puede explicar.

## Etapa 6 · En casa

Primero propón una columna con sentido de negocio y gánale al modelo de cuatro columnas en los mismos 50 mercados. Luego abre el dataset de CO₂ de años anteriores y mira qué le pasa a un coeficiente cuando las columnas son casi copias.

In [ ]:
# Pega aquí el código que te entregue tu asistente.
# Pista de prompt: quiero probar una columna nueva porque [tu razón de negocio]. Agrégala al modelo de cuatro
# columnas, entrena con train y mide en los mismos 50 de test. No pruebes otras columnas por tu cuenta.

In [ ]:
co2 = pd.read_csv("consumo-combustible-co2.csv")
COLS = ["ENGINESIZE", "CYLINDERS", "FUELCONSUMPTION_CITY", "FUELCONSUMPTION_HWY", "FUELCONSUMPTION_COMB", "FUELCONSUMPTION_COMB_MPG"]

# Pega aquí el código que te entregue tu asistente.
# Pista de prompt: toma seis mitades distintas (random_state de 0 a 5), estandariza, y ajusta sin freno, con
# Ridge alpha 50 y con Lasso alpha 1. Dame una tabla con el coeficiente de FUELCONSUMPTION_COMB en cada mitad.

**Decisión para el memo.** ¿Cuándo le recomendarías a Daniela Ridge, cuándo Lasso y cuándo ninguno?